# Kafka Learning Notes — Part 1: Topics, Partitions, and Offsets

## 1. What Is Kafka?

Apache Kafka is a **distributed event-streaming platform** used to publish, store, and consume streams of messages or events.

Its main components include:

- **Producers:** Applications that publish events to Kafka.
- **Topics:** Named streams that organize related events.
- **Consumers:** Applications that read and process events.
- **Brokers:** Servers that store events and handle producer and consumer requests.

## 2. Topics and Partitions

A Kafka topic groups related events, such as orders, payments, or application logs.

Each topic contains **one or more partitions**, such as partition 0, partition 1, partition 2, and partition 3. Each partition is an ordered log of records, with new records appended to its end.

Kafka preserves the order of records **within a partition**, but it does not guarantee a single order across all partitions in a topic.

A topic is better understood as a **partitioned event log** than as a database without a schema. Kafka stores serialized keys and values, while applications define their data formats and may use schemas.

## 3. Message Keys and Partition Assignment

A producer can include a **key** with each message, such as a customer ID or an order ID.

### Messages with a Key

With the standard key-based partitioning strategy, messages with the same key go to the same partition, provided the partition count and partitioning strategy remain unchanged.

For example, messages with the key `customer-123` are routed to the same partition, keeping that customer's events together.

Different keys can also map to the same partition.

### Messages Without a Key

When the key is `null`, partition assignment depends on the producer client and configuration.

Modern Kafka Java producers use **sticky partitioning behavior** by default for unkeyed records. They keep records together in a partition to improve batching before selecting another partition.

**Round-robin partitioning** is an available strategy, but it is not always the default for messages without a key.

## 4. Connecting Multiple Producers and Consumers

Kafka reduces the need for direct connections between every producer and consumer.

For example, producers A and B can publish events to a topic, while notification, analytics, and auditing applications consume those events independently.

Producers do not need to know which applications consume their events.

### Consumer Groups

- **Different consumer groups** can independently read the same events.
- **Consumers within a standard consumer group** share the work by dividing partitions among themselves.

Kafka supports event-driven communication, while applications implement any required workflow orchestration.

## 5. Horizontal Scaling

Kafka supports **horizontal scaling** by distributing partitions across brokers.

Consumers can also process different partitions in parallel. Within a standard consumer group, each partition is assigned to one consumer at a time.

For example, a topic with four partitions supports up to four consumers actively processing those partitions within the same group. Additional consumers in that group remain idle until a partition becomes available.

## 6. Offsets

An **offset** identifies a record's position within a partition.

Offsets are specific to each partition. For example, partition 0 and partition 1 can both contain a record at offset 5; those offsets identify different records.

Consumers use offsets to track their progress and can return to earlier offsets to replay records that Kafka still retains.

## 7. Immutability and Data Retention

Kafka records are **immutable**: once written, an existing record cannot be edited in place. A producer represents a change by publishing a new record.

However, immutability does not mean that records are stored forever:

- **Retention policies** can remove older records.
- **Log compaction** can remove older records for a key while retaining its latest value.
- Removed records leave gaps in the offsets. Remaining offsets are not renumbered, and removed offsets are not reused within that partition's lifetime.

Reading a record does not automatically delete it.

Kafka allows consumers to read and replay retained records by offset, but it does not directly provide ordinary database-style queries over a topic's records.

# Kafka Learning Notes — Part 2: Brokers, Consumer Assignment, Serialization, and Recovery

## 1. Kafka Brokers and Clusters

A **Kafka broker** is a server process that stores partition replicas and handles requests from producers and consumers.

A broker can run on infrastructure such as an AWS EC2 instance, a physical server, or a container. The instance provides the computing resources; the Kafka broker is the software running on it.

A **Kafka cluster** contains cooperating Kafka servers. A single broker can host partitions from multiple topics.

## 2. Consumers and Partition Assignment

Within a **standard consumer group**:

- Each partition is assigned to one consumer at a time.
- One consumer can read from multiple partitions.
- Consumers share the work of processing a topic.
- If there are more consumers than partitions, some consumers remain idle.

For example, a topic with four partitions and two consumers might have this assignment:

| Consumer | Assigned partitions |
|---|---|
| Consumer A | Partition 0, Partition 1 |
| Consumer B | Partition 2, Partition 3 |

A separate consumer group can independently read all four partitions.

Therefore, multiple consumers can read the same partition when they belong to different groups.

## 3. Serialization and Deserialization

**Serialization** converts application data into bytes before a producer sends it to Kafka.

**Deserialization** converts those bytes back into usable data after a consumer receives them.

Examples include integer and string serializers, as well as serializers for formats such as Avro.

Keys and values have separate serializer and deserializer configurations.

### Can a Topic's Data Format Change?

Kafka itself stores bytes and does not permanently assign a topic an integer, string, or Avro data type.

However, consumers must understand the format that producers write. Changing formats without updating consumers can cause deserialization errors.

A schema registry can manage schema versions and enforce compatibility rules. Compatible schema changes can often happen within the same topic.

For an incompatible format change, creating a new topic can simplify migration, but it is not a universal Kafka requirement. Existing records retain their original encoding.

## 4. Consumer Offset Commits

A consumer tracks its current reading position for each assigned partition.

An **offset commit** saves a checkpoint for a consumer group and topic-partition. It normally records the offset of the next record to consume.

For example:

- The consumer successfully processes records through offset 24.
- It commits offset 25.
- After restarting, it can resume from offset 25, provided that offset is still available.

Consumers can commit offsets automatically or explicitly through application code.

### Why Commit Timing Matters

- Processing records before committing can cause repeated processing if the consumer crashes before the commit succeeds.
- Committing before processing finishes can cause unfinished records to be skipped after recovery.

A commit alone does not guarantee exactly-once processing.

### Consumer Failure vs. Broker Failure

After a consumer restart or partition reassignment, committed offsets provide a recovery checkpoint.

If only a broker fails, a running consumer can discover the new partition leader and continue from its current position. It does not necessarily reload its committed offset.

## 5. Replication and Failover

**Replication** stores copies of the same partition on different brokers.

A replication factor of 3 means that each partition has three replicas in total: one leader and two followers.

For example:

| Broker | Replica of Partition 0 |
|---|---|
| Broker 1 | Leader |
| Broker 2 | Follower |
| Broker 3 | Follower |

The leader handles writes, and followers replicate its data. Consumers normally read from the leader, although follower reads can be configured.

If the leader's broker fails or undergoes maintenance, an eligible replica can become the new leader.

Another partition does not replace Partition 0. A replica of Partition 0 takes over its leadership.

Replication factor 3 is a common production choice. Availability and durability also depend on replica health, producer acknowledgments, and settings such as `min.insync.replicas`. Replication does not guarantee uninterrupted operation under every failure.

## 6. Bootstrap Servers and Metadata Discovery

Producers and consumers use `bootstrap.servers` as an initial list of broker addresses.

The client connects to an available bootstrap broker and requests cluster metadata, including broker addresses, topic partitions, and their leaders.

It then connects to the appropriate brokers for its operations:

- Producers send records to partition leaders.
- Consumers fetch records from the brokers serving their assigned partitions.
- Consumer-group coordination and offset commits use the group's coordinator.

The bootstrap broker is an entry point, not a gateway through which all traffic must pass.

Clients refresh metadata when needed, including after leadership changes. Providing multiple bootstrap addresses helps clients connect when one broker is unavailable.

## 7. Consumer Groups

A **consumer group** is a collection of consumers that cooperate to process records from subscribed topics. Consumers join the same group by using the same `group.id`.

### Partition Assignment Within a Group

In a standard consumer group:

- Each partition is assigned to one consumer at a time.
- One consumer can handle multiple partitions.
- Adding consumers increases parallelism up to the number of available partitions.
- Consumers beyond that number remain idle.

For example, a topic with four partitions and three consumers might have this assignment:

| Consumer | Assigned partitions |
|---|---|
| Consumer A | Partition 0, Partition 1 |
| Consumer B | Partition 2 |
| Consumer C | Partition 3 |

### Independent Consumer Groups

Different consumer groups can read the same topic independently, maintaining their own progress.

For example, an `orders` topic could serve:

- **notification-group:** Sends order notifications.
- **analytics-group:** Calculates order metrics.
- **audit-group:** Records order events for auditing.

Each group can process the same events at its own pace. Consuming an event in one group does not remove it or advance another group's offsets.

### Rebalancing

A **rebalance** redistributes partition assignments when group membership or subscribed partitions change—for example, when a consumer joins, leaves, or fails.

A consumer receiving a partition normally resumes from that group's committed offset for the partition.

## 8. Consumer Offsets

An **offset** identifies a record's position within a partition. Consumer progress is tracked separately for each topic-partition.

Two partitions can both contain offset 10, but those offsets refer to different records.

### Current Position vs. Committed Offset

| Concept | Meaning |
|---|---|
| Current position | The offset of the next record the consumer expects to fetch; it advances as records are returned to the application. |
| Committed offset | The saved recovery position for a consumer group and topic-partition. |

The current position does not necessarily mean that application processing has finished.

For example, if an application has successfully processed all records through offset 24, it can commit offset 25 as its next recovery position.

### Where Are Committed Offsets Stored?

Kafka stores standard consumer-group offset commits in an internal topic named `__consumer_offsets`.

A committed offset is associated with:

- Consumer group
- Topic
- Partition

It belongs to the group's progress for that partition, rather than permanently belonging to a particular consumer.

### Automatic and Manual Commits

- **Automatic commits:** The consumer client periodically commits offsets when automatic committing is enabled. This does not independently verify that business processing has completed.
- **Manual commits:** The application explicitly decides when to commit, typically after successful processing.

If processing succeeds but the consumer crashes before committing, those records may be processed again after recovery.

### Starting Without a Valid Committed Offset

The `auto.offset.reset` configuration controls what happens when there is no committed offset or the saved offset is no longer available.

Common options include:

| Setting | Behavior |
|---|---|
| `earliest` | Starts from the earliest record still retained in the partition. |
| `latest` | Starts from the end of the partition, waiting for new records. |
| `none` | Raises an error instead of automatically choosing a position. |

This setting does not override an existing valid committed offset during normal startup.

### Consumer Lag

**Consumer lag** indicates how far a consumer group is behind a partition's available records.

A common offset-based calculation is:

`Lag = Log-end offset − Committed offset`

For example, if the log-end offset is 100 and the committed offset is 80, the reported lag is 20 offset positions.

Lag is useful for monitoring whether consumers are keeping up. It is not always an exact count of unfinished business operations because offset gaps, transactions, and commit timing can affect its interpretation.